# Detector de Spam en Correos

**Evaluación Grupal 1 · Lenguaje de Programación II · Semestre 2026-2**

---

### Equipo de trabajo

**Grupo 5** · Universidad Nacional Agraria La Molina · Facultad de Economía y Planificación

| N° | Integrante | Usuario |
|:--:|------------|:--:|
| 1 | Huamani Diestra, Miriam Vanessa | MVanessaHD |
| 2 | Meza Asto, Angel David | AngelMol0810 |
| 3 | Chavez Cuzcano, Victor Javier | jchest771-stack |
| 4 | Alva Aquino, Brian | blolchevere-ctrl |

**Docente:** Vargas Paredes, Ana Cecilia

---

**Contenido:** 
1. Descripción del sistema 
2. Arquitectura 
3. Implementación 
4. Ejecución de prueba 
5. Menú interactivo 
6. Informe de clases 
7. Conceptos de POO 
8. Conclusiones


## 1. Descripción del sistema

Simulación de un **filtro de spam**. Cada correo se revisa con un conjunto de *reglas*; cada regla aporta puntos de "sospecha". El detector suma los puntos y, si el total alcanza un **umbral**, marca el correo como spam y guarda los **motivos** que lo explican.

**Funcionalidades**

- Agregar correos a una bandeja.
- Analizar cada correo con varias reglas.
- Mostrar el puntaje y los motivos de cada decisión.
- Listar correos spam y normales.
- Cambiar el umbral de sensibilidad.
- Sumar una regla opcional de **Naive Bayes** básico, que aprende de ejemplos y se integra como una regla más.

## 2. Arquitectura y diseño

```
Bandeja ──contiene──▶ Correo ◀──analiza── DetectorSpam ──tiene──▶ Regla
                                                                    ▲
         ┌──────────────────┬─────────────────┬──────────────┬───┴────────────┬────────────────┐
 ReglaPalabrasClave   ReglaRemitente   ReglaEnlaces   ReglaMayusculas   ReglaNaiveBayes
```

| Relación | Tipo | Significado |
|---|---|---|
| `Bandeja` → `Correo` | Composición | La bandeja guarda una lista de correos |
| `DetectorSpam` → `Regla` | Composición | El detector guarda una lista de reglas |
| `DetectorSpam` ⇢ `Correo` | Dependencia | El detector analiza y modifica un correo |
| `ReglaXxx` ⇒ `Regla` | Herencia | Cada regla especializa a la clase base |

**Flujo de una decisión:** `Bandeja.agregar(correo)` → `DetectorSpam.analizar(correo)` → cada `Regla.evaluar(correo)` devuelve `(puntaje, motivo)` → el detector suma, guarda el puntaje en el correo y lo marca como spam o normal → `Bandeja.listar_spam()` y `Bandeja.resumen()` muestran el resultado.

**Contrato entre clases**

| Clase | Método | Recibe | Devuelve |
|---|---|---|---|
| `Regla` y sus hijas | `evaluar(correo)` | un `Correo` | tupla `(puntaje, motivo)` |
| `DetectorSpam` | `analizar(correo)` | un `Correo` | `True` si es spam |
| `Bandeja` | `listar_spam()`, `listar_normales()` | — | lista de `Correo` |

## 3. Implementación

El orden de las celdas importa: cada clase se define antes de usarse.

### 3.1 Clase `Correo`
Representa un mensaje y guarda el resultado de su análisis.

In [19]:
class Correo:
    """Un correo con sus datos y el resultado del análisis de spam.

    Los atributos son privados (encapsulamiento). Se leen mediante propiedades
    y el puntaje se valida en su setter.
    """

    def __init__(self, remitente, asunto, cuerpo):
        self.__remitente = remitente
        self.__asunto = asunto
        self.__cuerpo = cuerpo
        self.__puntaje = 0
        self.__es_spam = False
        self.__motivos = []

    @property
    def remitente(self):
        return self.__remitente

    @property
    def asunto(self):
        return self.__asunto

    @property
    def cuerpo(self):
        return self.__cuerpo

    @property
    def puntaje(self):
        return self.__puntaje

    @puntaje.setter
    def puntaje(self, valor):
        if valor < 0:
            raise ValueError("El puntaje no puede ser negativo")
        self.__puntaje = valor

    @property
    def es_spam(self):
        return self.__es_spam

    @property
    def motivos(self):
        # Devuelve una copia para no exponer la lista interna
        return list(self.__motivos)

    def agregar_motivo(self, motivo):
        """Registra por qué una regla sospechó de este correo."""
        self.__motivos.append(motivo)

    def limpiar_motivos(self):
        """Borra los motivos de un análisis anterior."""
        self.__motivos.clear()

    def marcar_como_spam(self):
        self.__es_spam = True

    def marcar_como_normal(self):
        self.__es_spam = False

    def __str__(self):
        estado = "SPAM" if self.__es_spam else "NORMAL"
        return f"[{estado}] ({self.__puntaje} pts) {self.__remitente} - {self.__asunto}"

**Resumen de métodos de `Correo`**

| Método | Qué hace |
|---|---|
| `__init__(remitente, asunto, cuerpo)` | Crea el correo con sus datos; deja el puntaje en 0, no spam y sin motivos |
| `remitente`, `asunto`, `cuerpo` | Propiedades de solo lectura: devuelven los datos originales |
| `puntaje` | Lee el puntaje; al asignarlo, valida que no sea negativo |
| `es_spam` | Indica si el correo fue marcado como spam |
| `motivos` | Devuelve una copia de la lista de motivos (la interna no se expone) |
| `agregar_motivo(motivo)` | Registra por qué una regla sospechó del correo |
| `limpiar_motivos()` | Borra los motivos de un análisis anterior |
| `marcar_como_spam()`, `marcar_como_normal()` | Cambian el estado del correo |
| `__str__()` | Texto legible: estado, puntaje, remitente y asunto |

### 3.2 Clase base `Regla` y sus hijas (herencia)
`Regla` es una clase **abstracta**: define qué debe saber hacer cualquier regla (`evaluar`) pero no se puede instanciar. Cada regla hija **hereda** `nombre` y `peso` y la especializa implementando `evaluar(correo)`, que devuelve la tupla `(puntaje, motivo)` o `(0, "")` si la regla no se cumple. Como el detector solo llama a `evaluar`, puede usar cualquier regla sin conocer su tipo (**polimorfismo**).

In [20]:
import math
import re
import unicodedata
from abc import ABC, abstractmethod


def normalizar(texto):
    """Pasa a minúsculas y quita tildes, para que 'PREMIÓ' y 'premio' coincidan."""
    descompuesto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")


class Regla(ABC):
    """Clase base abstracta: cada regla aporta puntos de sospecha a un correo.

    Es abstracta porque no tiene sentido crear una 'Regla' genérica: solo se
    usan sus hijas. Cada hija DEBE implementar ``evaluar`` (polimorfismo).
    """

    def __init__(self, nombre, peso):
        if peso <= 0:
            raise ValueError("El peso de una regla debe ser positivo")
        self.__nombre = nombre
        self.__peso = peso

    @property
    def nombre(self):
        return self.__nombre

    @property
    def peso(self):
        return self.__peso

    @abstractmethod
    def evaluar(self, correo):
        """Recibe un Correo y devuelve la tupla (puntaje, motivo).

        Si la regla no se cumple devuelve (0, "").
        """

    def __str__(self):
        return f"{self.nombre} (peso {self.peso})"


class ReglaPalabrasClave(Regla):
    """Suma ``peso`` por cada palabra sospechosa distinta (acepta plurales)."""

    PALABRAS_POR_DEFECTO = ("gratis", "premio", "urgente", "ganaste", "oferta")

    def __init__(self, palabras=None, peso=2):
        super().__init__("Palabras clave", peso)
        self.__palabras = tuple(palabras or self.PALABRAS_POR_DEFECTO)

    def evaluar(self, correo):
        texto = normalizar(f"{correo.asunto} {correo.cuerpo}")
        encontradas = [
            p for p in self.__palabras
            if re.search(rf"\b{re.escape(normalizar(p))}\w*", texto)
        ]
        if not encontradas:
            return 0, ""
        return len(encontradas) * self.peso, f"{self.nombre}: contiene {', '.join(encontradas)}"


class ReglaRemitente(Regla):
    """El dominio del remitente está en una lista negra (incluye subdominios)."""

    LISTA_NEGRA_POR_DEFECTO = ("promo-ya.com", "ganadores.net", "gratis.xyz")

    def __init__(self, lista_negra=None, peso=5):
        super().__init__("Remitente sospechoso", peso)
        self.__lista_negra = tuple(d.lower() for d in (lista_negra or self.LISTA_NEGRA_POR_DEFECTO))

    def evaluar(self, correo):
        dominio = correo.remitente.split("@")[-1].lower()
        for bloqueado in self.__lista_negra:
            # igual al dominio, o subdominio (mail.promo-ya.com); NO 'xpromo-ya.com'
            if dominio == bloqueado or dominio.endswith("." + bloqueado):
                return self.peso, f"{self.nombre}: remitente en lista negra ({bloqueado})"
        return 0, ""


class ReglaEnlaces(Regla):
    """El cuerpo del correo tiene demasiados enlaces."""

    def __init__(self, max_enlaces=2, peso=2):
        super().__init__("Enlaces", peso)
        self.__max_enlaces = max_enlaces

    def evaluar(self, correo):
        enlaces = re.findall(r"https?://\S+", correo.cuerpo, flags=re.IGNORECASE)
        if len(enlaces) > self.__max_enlaces:
            return self.peso, f"{self.nombre}: tiene {len(enlaces)} enlaces"
        return 0, ""


class ReglaMayusculas(Regla):
    """El asunto está en mayúsculas o tiene muchos signos de exclamación."""

    def __init__(self, peso=3):
        super().__init__("Mayúsculas y exclamaciones", peso)

    def evaluar(self, correo):
        asunto = correo.asunto
        letras = [c for c in asunto if c.isalpha()]
        motivos = []
        if len(letras) > 3 and asunto.isupper():
            motivos.append("asunto en mayúsculas")
        if asunto.count("!") >= 3:
            motivos.append("muchos signos de exclamación")
        if not motivos:
            return 0, ""
        return self.peso, f"{self.nombre}: {', '.join(motivos)}"


class ReglaNaiveBayes(Regla):
    """Regla opcional que APRENDE de ejemplos (Naive Bayes multinomial básico).

    Antes de entrenar (o si solo vio spam o solo normales) no opina: devuelve
    (0, ""). Una vez entrenada, si la probabilidad de spam del correo es
    mayor o igual que ``umbral_prob``, suma ``peso`` puntos como cualquier
    otra regla.
    """

    def __init__(self, peso=3, umbral_prob=0.5, alfa=1.0):
        super().__init__("Naive Bayes", peso)
        if not 0 < umbral_prob < 1:
            raise ValueError("umbral_prob debe estar entre 0 y 1")
        if alfa <= 0:
            raise ValueError("alfa debe ser mayor que 0")
        self.__umbral_prob = umbral_prob
        self.__alfa = alfa                           # suavizado de Laplace
        self.__n_correos = {True: 0, False: 0}       # correos vistos por clase
        self.__total_palabras = {True: 0, False: 0}  # palabras vistas por clase
        self.__frecuencias = {True: {}, False: {}}   # palabra -> veces, por clase
        self.__vocabulario = set()

    @property
    def entrenada(self):
        """True si ya vio al menos un correo spam y uno normal."""
        return self.__n_correos[True] > 0 and self.__n_correos[False] > 0

    def __palabras(self, correo):
        texto = normalizar(f"{correo.asunto} {correo.cuerpo}")
        return [p for p in re.findall(r"[a-z0-9]+", texto) if len(p) >= 3]

    def entrenar(self, ejemplos):
        """Aprende de una lista de pares (correo, es_spam). Se puede llamar varias veces."""
        for correo, es_spam in ejemplos:
            clase = bool(es_spam)
            self.__n_correos[clase] += 1
            for palabra in self.__palabras(correo):
                self.__frecuencias[clase][palabra] = self.__frecuencias[clase].get(palabra, 0) + 1
                self.__total_palabras[clase] += 1
                self.__vocabulario.add(palabra)

    def probabilidad_spam(self, correo):
        """Devuelve P(spam | correo) entre 0 y 1 (0.5 si aún no está entrenada)."""
        if not self.entrenada:
            return 0.5
        total_correos = self.__n_correos[True] + self.__n_correos[False]
        tamano_vocabulario = len(self.__vocabulario)
        log_prob = {}
        for clase in (True, False):
            lp = math.log(self.__n_correos[clase] / total_correos)
            denominador = self.__total_palabras[clase] + self.__alfa * tamano_vocabulario
            for palabra in self.__palabras(correo):
                if palabra in self.__vocabulario:
                    veces = self.__frecuencias[clase].get(palabra, 0)
                    lp += math.log((veces + self.__alfa) / denominador)
            log_prob[clase] = lp
        diferencia = max(-700, min(700, log_prob[False] - log_prob[True]))
        return 1 / (1 + math.exp(diferencia))

    def evaluar(self, correo):
        if not self.entrenada:
            return 0, ""
        prob = self.probabilidad_spam(correo)
        if prob >= self.__umbral_prob:
            return self.peso, f"{self.nombre}: probabilidad de spam {prob:.0%}"
        return 0, ""

**Resumen de `Regla` y sus hijas**

| Clase | Hereda de | Qué evalúa | Puntos que aporta |
|---|---|---|---|
| `Regla` | `ABC` | Clase base abstracta: guarda `nombre` y `peso` (debe ser mayor que 0) y obliga a las hijas a implementar `evaluar` | — |
| `ReglaPalabrasClave` | `Regla` | Palabras sospechosas en asunto y cuerpo (ignora mayúsculas y tildes; acepta plurales) | `peso` por cada palabra distinta (por defecto 2) |
| `ReglaRemitente` | `Regla` | Dominio del remitente en una lista negra (también sus subdominios) | `peso` (por defecto 5) |
| `ReglaEnlaces` | `Regla` | Más de `max_enlaces` enlaces en el cuerpo (por defecto 2) | `peso` (por defecto 2) |
| `ReglaMayusculas` | `Regla` | Asunto en mayúsculas o con 3 o más signos de exclamación | `peso` (por defecto 3) |
| `ReglaNaiveBayes` | `Regla` | Probabilidad de spam **aprendida de ejemplos** (Naive Bayes básico). Sin entrenar no opina | `peso` (por defecto 3) si la probabilidad es mayor o igual que `umbral_prob` (por defecto 0.5) |

**Métodos de `Regla` (los heredan todas las hijas)**

| Método | Qué hace |
|---|---|
| `__init__(nombre, peso)` | Guarda el nombre y el peso; valida que el peso sea positivo |
| `nombre`, `peso` | Propiedades de solo lectura |
| `evaluar(correo)` | **Abstracto.** Cada hija lo implementa y devuelve `(puntaje, motivo)`; si no se cumple, `(0, "")` |
| `__str__()` | Texto legible: nombre y peso |

**Métodos propios de `ReglaNaiveBayes`**

| Método | Qué hace |
|---|---|
| `entrenar(ejemplos)` | Aprende de una lista de pares `(correo, es_spam)`; se puede llamar varias veces |
| `entrenada` | `True` si ya vio al menos un correo spam y uno normal |
| `probabilidad_spam(correo)` | Devuelve P(spam \| correo) entre 0 y 1 (0.5 si aún no está entrenada) |

### 3.3 Clase `DetectorSpam`
Es el motor: guarda una lista de reglas, las aplica a un correo y decide si es spam. **No conoce los tipos concretos de reglas** (polimorfismo).

In [21]:
class DetectorSpam:
    """Contiene una lista de reglas y decide si un correo es spam."""

    def __init__(self, umbral=5):
        self.__reglas = []
        self.__umbral = umbral

    @property
    def umbral(self):
        return self.__umbral

    @umbral.setter
    def umbral(self, valor):
        if valor <= 0:
            raise ValueError("El umbral debe ser mayor que 0")
        self.__umbral = valor

    @property
    def reglas(self):
        return list(self.__reglas)

    def agregar_regla(self, regla):
        """Agrega una regla. Cualquier objeto que herede de Regla sirve."""
        if not isinstance(regla, Regla):
            raise TypeError("Solo se pueden agregar objetos Regla")
        self.__reglas.append(regla)

    def analizar(self, correo):
        """Evalúa el correo con todas las reglas y lo marca si supera el umbral."""
        correo.limpiar_motivos()
        total = 0
        for regla in self.__reglas:  # polimorfismo: no importa el tipo de regla
            puntaje, motivo = regla.evaluar(correo)
            total += puntaje
            if motivo:
                correo.agregar_motivo(motivo)

        correo.puntaje = total
        if total >= self.__umbral:
            correo.marcar_como_spam()
        else:
            correo.marcar_como_normal()
        return correo.es_spam

**Resumen de métodos de `DetectorSpam`**

| Método | Qué hace |
|---|---|
| `__init__(umbral)` | Crea el detector con una lista de reglas vacía y el umbral indicado |
| `umbral` (propiedad y setter) | Lee el umbral; al asignarlo, valida que sea mayor que 0 |
| `reglas` | Devuelve una copia de la lista de reglas |
| `agregar_regla(regla)` | Agrega una regla; valida que sea un objeto `Regla` (o de una clase hija) |
| `analizar(correo)` | Aplica todas las reglas, suma los puntajes, guarda puntaje y motivos en el correo y lo marca spam o normal según el umbral |

### 3.4 Clase `Bandeja`
Almacena los correos y responde consultas. **No analiza** (eso es del detector) ni imprime (eso es del menú).

In [22]:
class Bandeja:
    """Almacena los correos y responde consultas.

    No analiza (eso es del DetectorSpam) ni imprime (eso es del menú):
    solo guarda y devuelve datos.
    """

    def __init__(self):
        self.__correos = []

    @property
    def correos(self):
        # Copia de la lista: nadie puede modificar la bandeja desde afuera
        return list(self.__correos)

    def agregar(self, correo):
        """Agrega un correo. Solo acepta objetos Correo."""
        if not isinstance(correo, Correo):
            raise TypeError("Solo se pueden agregar objetos Correo")
        self.__correos.append(correo)

    def listar_spam(self):
        return [c for c in self.__correos if c.es_spam]

    def listar_normales(self):
        return [c for c in self.__correos if not c.es_spam]

    def resumen(self):
        """Devuelve un diccionario con las estadísticas de la bandeja."""
        total = len(self.__correos)
        spam = len(self.listar_spam())
        return {
            "total": total,
            "spam": spam,
            "normales": total - spam,
            "porcentaje_spam": round(spam / total * 100, 1) if total else 0.0,
        }

    def __len__(self):
        return len(self.__correos)

**Resumen de métodos de `Bandeja`**

| Método | Qué hace |
|---|---|
| `__init__()` | Crea la bandeja con una lista de correos vacía |
| `correos` | Devuelve una copia de la lista de correos |
| `agregar(correo)` | Agrega un correo; lanza `TypeError` si no es un `Correo` |
| `listar_spam()` | Devuelve los correos marcados como spam |
| `listar_normales()` | Devuelve los correos no marcados como spam |
| `resumen()` | Devuelve un diccionario: `total`, `spam`, `normales` y `porcentaje_spam` |
| `__len__()` | Permite usar `len(bandeja)` |

> Un correo que aún no fue analizado figura como *normal*, porque `es_spam` empieza en `False`.

## 4. Ejecución de prueba

Se usan **todas** las clases en secuencia.

**Paso 1: crear el detector y registrar las reglas**

In [23]:
def ejemplos_de_entrenamiento():
    """Correos etiquetados con los que aprende la regla Naive Bayes."""
    spam = [
        ("promos@ofertas-ya.com", "Gana dinero rápido", "Haz clic ahora y gana dinero rápido sin esfuerzo"),
        ("sorteo@ganadores.net", "Felicidades, ganaste el sorteo", "Reclama ahora tu premio del sorteo exclusivo"),
        ("ventas@descuentos.com", "Oferta limitada", "Descuento exclusivo compra ya la oferta limitada"),
        ("seguridad@banco-falso.com", "Tu cuenta fue bloqueada", "Verifica urgente tu cuenta bloqueada haz clic"),
        ("prestamos@credito-facil.com", "Préstamo inmediato", "Préstamo inmediato sin requisitos solicita ahora"),
        ("info@gratis.xyz", "Suscríbete gratis", "Promoción gratis suscríbete ahora y gana un premio"),
    ]
    normales = [
        ("profe@lamolina.edu.pe", "Revisión del informe", "Adjunto la rúbrica de evaluación del informe del curso"),
        ("jefe@empresa.pe", "Reunión de coordinación", "La reunión del equipo será mañana para revisar el avance"),
        ("secretaria@lamolina.edu.pe", "Horario de laboratorio", "Te comparto el horario del laboratorio de programación"),
        ("amigo@correo.com", "Avance del trabajo grupal", "Terminé mi parte del trabajo grupal, revisa el código"),
        ("notas@lamolina.edu.pe", "Notas del curso", "Ya están publicadas las notas de la práctica del curso"),
        ("familia@correo.com", "Cena del domingo", "Nos vemos el domingo en casa para almorzar en familia"),
    ]
    return ([(Correo(*datos), True) for datos in spam]
            + [(Correo(*datos), False) for datos in normales])


def crear_detector(umbral=5):
    """Arma un detector con las 4 reglas fijas y la regla Naive Bayes entrenada."""
    detector = DetectorSpam(umbral)
    # umbral_prob=0.6: una probabilidad de exactamente 50% significa "no conozco estas
    # palabras", por eso se exige un poco más de confianza para sumar puntos
    bayes = ReglaNaiveBayes(peso=3, umbral_prob=0.6)
    for regla in (ReglaPalabrasClave(), ReglaRemitente(), ReglaEnlaces(),
                  ReglaMayusculas(), bayes):
        detector.agregar_regla(regla)
    bayes.entrenar(ejemplos_de_entrenamiento())
    return detector, bayes


detector, bayes = crear_detector(umbral=5)

print(f"Umbral del detector: {detector.umbral} puntos")
print("Reglas registradas:")
for regla in detector.reglas:
    print(f"  - {regla}")
print(f"Naive Bayes entrenada: {bayes.entrenada}")

Umbral del detector: 5 puntos
Reglas registradas:
  - Palabras clave (peso 2)
  - Remitente sospechoso (peso 5)
  - Enlaces (peso 2)
  - Mayúsculas y exclamaciones (peso 3)
  - Naive Bayes (peso 3)
Naive Bayes entrenada: True


**Paso 2: crear la bandeja y cargar correos de ejemplo** (spam claros, normales y casos dudosos)

In [24]:
def correos_de_ejemplo():
    return [
        Correo("ventas@promo-ya.com", "¡¡¡GANASTE UN PREMIO GRATIS!!!",
               "Haz clic: http://a.co http://b.co http://c.co"),
        Correo("info@ganadores.net", "URGENTE: reclama tu premio",
               "Has sido seleccionado. Reclama aquí: http://premio.co/a http://premio.co/b"),
        Correo("profe.vargas@lamolina.edu.pe", "Revisión del informe",
               "Adjunto la rúbrica de evaluación del trabajo grupal."),
        Correo("compa@correo.com", "Reunión del grupo 5",
               "Nos vemos mañana para avanzar con el informe de POO."),
        Correo("soporte@tienda.com", "Oferta de temporada",
               "Descuento en tu próxima compra."),
        Correo("secretaria@lamolina.edu.pe", "URGENTE: confirma tu asistencia",
               "Necesitamos confirmar tu asistencia a la reunión de coordinación."),
        Correo("boletin@mail.promo-ya.com", "Boletín semanal",
               "Resumen de novedades de la semana."),
        Correo("hola@xpromo-ya.com", "Boletín semanal",
               "Resumen de novedades de la semana."),
        Correo("ventas@tiendaperu.com", "Ofertas gratis esta semana",
               "Visita nuestra tienda: http://tienda.pe"),
    ]


bandeja = Bandeja()
for correo in correos_de_ejemplo():
    bandeja.agregar(correo)

print(f"Correos cargados en la bandeja: {len(bandeja)}")

Correos cargados en la bandeja: 9


**Paso 3: analizar todos los correos**

In [25]:
for correo in bandeja.correos:
    detector.analizar(correo)   # cada llamada guarda puntaje, motivos y estado en el correo

print(f"{'Puntaje':>7}  {'Veredicto':<9}  Remitente")
print("-" * 60)
for correo in bandeja.correos:
    veredicto = "SPAM" if correo.es_spam else "NORMAL"
    print(f"{correo.puntaje:>7}  {veredicto:<9}  {correo.remitente}")

Puntaje  Veredicto  Remitente
------------------------------------------------------------
     19  SPAM       ventas@promo-ya.com
     12  SPAM       info@ganadores.net
      0  NORMAL     profe.vargas@lamolina.edu.pe
      0  NORMAL     compa@correo.com
      5  SPAM       soporte@tienda.com
      2  NORMAL     secretaria@lamolina.edu.pe
      5  SPAM       boletin@mail.promo-ya.com
      0  NORMAL     hola@xpromo-ya.com
      7  SPAM       ventas@tiendaperu.com


**Paso 4: mostrar los resultados** (spam con sus motivos y correos normales)

In [26]:
def mostrar_lista(titulo, correos):
    print(f"=== {titulo} ({len(correos)}) ===")
    for correo in correos:
        print(correo)
        for motivo in correo.motivos:
            print(f"      - {motivo}")
    print()


mostrar_lista("CORREOS SPAM", bandeja.listar_spam())

=== CORREOS SPAM (5) ===
[SPAM] (19 pts) ventas@promo-ya.com - ¡¡¡GANASTE UN PREMIO GRATIS!!!
      - Palabras clave: contiene gratis, premio, ganaste
      - Remitente sospechoso: remitente en lista negra (promo-ya.com)
      - Enlaces: tiene 3 enlaces
      - Mayúsculas y exclamaciones: asunto en mayúsculas, muchos signos de exclamación
      - Naive Bayes: probabilidad de spam 100%
[SPAM] (12 pts) info@ganadores.net - URGENTE: reclama tu premio
      - Palabras clave: contiene premio, urgente
      - Remitente sospechoso: remitente en lista negra (ganadores.net)
      - Naive Bayes: probabilidad de spam 100%
[SPAM] (5 pts) soporte@tienda.com - Oferta de temporada
      - Palabras clave: contiene oferta
      - Naive Bayes: probabilidad de spam 93%
[SPAM] (5 pts) boletin@mail.promo-ya.com - Boletín semanal
      - Remitente sospechoso: remitente en lista negra (promo-ya.com)
[SPAM] (7 pts) ventas@tiendaperu.com - Ofertas gratis esta semana
      - Palabras clave: contiene gratis, ofe

In [27]:
mostrar_lista("CORREOS NORMALES", bandeja.listar_normales())

r = bandeja.resumen()
print("=== RESUMEN DE BANDEJA ===")
print(f"Total de correos : {r['total']}")
print(f"Spam             : {r['spam']}")
print(f"Normales         : {r['normales']}")
print(f"Porcentaje spam  : {r['porcentaje_spam']}%")

=== CORREOS NORMALES (4) ===
[NORMAL] (0 pts) profe.vargas@lamolina.edu.pe - Revisión del informe
[NORMAL] (0 pts) compa@correo.com - Reunión del grupo 5
[NORMAL] (2 pts) secretaria@lamolina.edu.pe - URGENTE: confirma tu asistencia
      - Palabras clave: contiene urgente
[NORMAL] (0 pts) hola@xpromo-ya.com - Boletín semanal

=== RESUMEN DE BANDEJA ===
Total de correos : 9
Spam             : 5
Normales         : 4
Porcentaje spam  : 55.6%


**Paso 5: demostrar herencia, polimorfismo y encapsulamiento**


In [28]:
ejemplo = bandeja.correos[0]   # el spam claro del primer lugar

print("HERENCIA: todas las reglas son instancias de Regla")
for regla in detector.reglas:
    print(f"  {type(regla).__name__:<20} isinstance(Regla) = {isinstance(regla, Regla)}")

print()
print("POLIMORFISMO: la misma llamada evaluar() se comporta distinto según la regla")
for regla in detector.reglas:
    puntaje, motivo = regla.evaluar(ejemplo)
    print(f"  {type(regla).__name__:<20} -> ({puntaje}, {motivo!r})")

print()
print("ENCAPSULAMIENTO: los datos se leen por propiedades, no por atributos públicos")
print(f"  ejemplo.puntaje = {ejemplo.puntaje}")
print(f"  ejemplo.motivos devuelve una copia (se puede modificar sin afectar al correo): "
      f"{ejemplo.motivos is not ejemplo.motivos}")
print(f"  atributos reales (name mangling): {[a for a in vars(ejemplo) if a.startswith('_Correo')][:3]} ...")

HERENCIA: todas las reglas son instancias de Regla
  ReglaPalabrasClave   isinstance(Regla) = True
  ReglaRemitente       isinstance(Regla) = True
  ReglaEnlaces         isinstance(Regla) = True
  ReglaMayusculas      isinstance(Regla) = True
  ReglaNaiveBayes      isinstance(Regla) = True

POLIMORFISMO: la misma llamada evaluar() se comporta distinto según la regla
  ReglaPalabrasClave   -> (6, 'Palabras clave: contiene gratis, premio, ganaste')
  ReglaRemitente       -> (5, 'Remitente sospechoso: remitente en lista negra (promo-ya.com)')
  ReglaEnlaces         -> (2, 'Enlaces: tiene 3 enlaces')
  ReglaMayusculas      -> (3, 'Mayúsculas y exclamaciones: asunto en mayúsculas, muchos signos de exclamación')
  ReglaNaiveBayes      -> (3, 'Naive Bayes: probabilidad de spam 100%')

ENCAPSULAMIENTO: los datos se leen por propiedades, no por atributos públicos
  ejemplo.puntaje = 19
  ejemplo.motivos devuelve una copia (se puede modificar sin afectar al correo): True
  atributos reales (name

*Encapsulamiento y validaciones:* intentar romper las reglas de la clase produce errores controlados.

In [29]:
def intentar(descripcion, accion):
    """Ejecuta una acción prohibida y muestra el error controlado que produce."""
    try:
        accion()
        print(f"  [sin error] {descripcion}")
    except Exception as error:
        print(f"  {descripcion}\n      -> {type(error).__name__}: {error}")


correo = Correo("a@b.com", "Asunto", "Cuerpo")

print("Intentos de romper las reglas de las clases:")
intentar("Correo: puntaje negativo", lambda: setattr(correo, "puntaje", -1))
intentar("Correo: cambiar el remitente (solo lectura)", lambda: setattr(correo, "remitente", "otro@x.com"))
intentar("Correo: acceder a un atributo privado", lambda: correo.__puntaje)
intentar("Regla: instanciar la clase abstracta", lambda: Regla("x", 1))
intentar("Regla: peso cero", lambda: ReglaEnlaces(peso=0))
intentar("ReglaNaiveBayes: umbral_prob fuera de (0, 1)", lambda: ReglaNaiveBayes(umbral_prob=2))
intentar("DetectorSpam: umbral cero", lambda: setattr(detector, "umbral", 0))
intentar("DetectorSpam: agregar algo que no es Regla", lambda: detector.agregar_regla("hola"))
intentar("Bandeja: agregar algo que no es Correo", lambda: bandeja.agregar("hola"))

Intentos de romper las reglas de las clases:
  Correo: puntaje negativo
      -> ValueError: El puntaje no puede ser negativo
  Correo: cambiar el remitente (solo lectura)
      -> AttributeError: property 'remitente' of 'Correo' object has no setter
  Correo: acceder a un atributo privado
      -> AttributeError: 'Correo' object has no attribute '__puntaje'
  Regla: instanciar la clase abstracta
      -> TypeError: Can't instantiate abstract class Regla without an implementation for abstract method 'evaluar'
  Regla: peso cero
      -> ValueError: El peso de una regla debe ser positivo
  ReglaNaiveBayes: umbral_prob fuera de (0, 1)
      -> ValueError: umbral_prob debe estar entre 0 y 1
  DetectorSpam: umbral cero
      -> ValueError: El umbral debe ser mayor que 0
  DetectorSpam: agregar algo que no es Regla
      -> TypeError: Solo se pueden agregar objetos Regla
  Bandeja: agregar algo que no es Correo
      -> TypeError: Solo se pueden agregar objetos Correo


## 5. Menú interactivo 

Menú en consola con `input()`.

**Para usar el menú de verdad** (con tu teclado), ejecuta la celda siguiente, quitando el `#` de la última línea.
Más abajo se **simula** a un usuario para dejar registradas sesiones de ejemplo.

In [30]:
class MenuInteractivo:
    """Menú de consola para usar el sistema. Es el único que usa input() y print()."""

    TITULO = """
===== DETECTOR DE SPAM =====
1. Agregar un correo (se analiza al instante)
2. Volver a analizar todos los correos
3. Ver correos SPAM (con motivos)
4. Ver correos NORMALES
5. Cambiar el umbral de sensibilidad
6. Entrenar Naive Bayes con la bandeja actual
7. Ver las reglas activas
8. Ver resumen de la bandeja
0. Salir"""

    TITULO_COMPACTO = ("Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | "
                       "5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir")

    def __init__(self, detector, bandeja, bayes=None, entrada=input, compacto=False):
        self.__detector = detector
        self.__bandeja = bandeja
        self.__bayes = bayes
        self.__entrada = entrada   # se puede reemplazar para simular al usuario
        self.__compacto = compacto  # True: muestra el menú en una sola línea

    # ---- utilidades internas -------------------------------------------
    def __pedir(self, mensaje):
        return self.__entrada(mensaje).strip()

    def __mostrar_correos(self, correos):
        if not correos:
            print("  (no hay correos)")
        for correo in correos:
            print(" ", correo)
            for motivo in correo.motivos:
                print(f"        - {motivo}")

    # ---- opciones --------------------------------------------------------
    def __agregar(self):
        remitente = self.__pedir("Remitente (ej. nombre@dominio.com): ")
        if "@" not in remitente:
            print("  Remitente inválido: debe contener '@'. No se agregó el correo.")
            return
        asunto = self.__pedir("Asunto: ")
        cuerpo = self.__pedir("Cuerpo: ")
        correo = Correo(remitente, asunto, cuerpo)
        self.__bandeja.agregar(correo)
        self.__detector.analizar(correo)
        print("  Resultado:", correo)
        for motivo in correo.motivos:
            print(f"        - {motivo}")

    def __reanalizar(self):
        for correo in self.__bandeja.correos:
            self.__detector.analizar(correo)
        print(f"  Se analizaron {len(self.__bandeja)} correos con umbral {self.__detector.umbral}.")

    def __cambiar_umbral(self):
        try:
            nuevo = int(self.__pedir(f"Nuevo umbral (actual {self.__detector.umbral}): "))
            antes = len(self.__bandeja.listar_spam())
            self.__detector.umbral = nuevo
        except ValueError as error:
            print(f"  Umbral no válido: {error}")
            return
        self.__reanalizar()
        despues = len(self.__bandeja.listar_spam())
        print(f"  Spam antes: {antes} | Spam ahora: {despues}")

    def __entrenar(self):
        if self.__bayes is None:
            print("  Este detector no tiene regla Naive Bayes.")
            return
        ejemplos = [(c, c.es_spam) for c in self.__bandeja.correos]
        self.__bayes.entrenar(ejemplos)
        print(f"  Naive Bayes aprendió de {len(ejemplos)} correos. Entrenada: {self.__bayes.entrenada}")

    def __resumen(self):
        r = self.__bandeja.resumen()
        print(f"  Total: {r['total']} | Spam: {r['spam']} | Normales: {r['normales']} "
              f"| % spam: {r['porcentaje_spam']}%")

    # ---- bucle principal -------------------------------------------------
    def ejecutar(self):
        while True:
            print(self.TITULO_COMPACTO if self.__compacto else self.TITULO)
            try:
                opcion = self.__pedir("Elige una opción: ")
            except (EOFError, StopIteration):
                print("\n(Sin más entradas, se cierra el menú)")
                break

            if opcion == "1":
                self.__agregar()
            elif opcion == "2":
                self.__reanalizar()
            elif opcion == "3":
                self.__mostrar_correos(self.__bandeja.listar_spam())
            elif opcion == "4":
                self.__mostrar_correos(self.__bandeja.listar_normales())
            elif opcion == "5":
                self.__cambiar_umbral()
            elif opcion == "6":
                self.__entrenar()
            elif opcion == "7":
                for regla in self.__detector.reglas:
                    print(f"  - {regla}")
            elif opcion == "8":
                self.__resumen()
            elif opcion == "0":
                print("Hasta luego.")
                break
            else:
                print("  Opción no válida, intenta de nuevo.")

In [31]:
detector_menu, bayes_menu = crear_detector()
bandeja_menu = Bandeja()
for c in correos_de_ejemplo():
    bandeja_menu.agregar(c)
    detector_menu.analizar(c)

# MenuInteractivo(detector_menu, bandeja_menu, bayes_menu).ejecutar()   # <- quita el '#' para usarlo

**Sesión de ejemplo** (las respuestas del usuario están simuladas y se muestran después de cada pregunta; el menú se imprime en modo compacto para ahorrar espacio). Se divide en 3 sesiones cortas sobre la misma bandeja.

**Sesión 1:** ver el resumen, agregar un correo spam y probar un remitente inválido.

In [32]:
def simular(*respuestas):
    """Devuelve una función que reemplaza a input() y responde con la lista dada."""
    it = iter(respuestas)

    def usuario(mensaje):
        respuesta = next(it)
        print(f"{mensaje}{respuesta}")
        return respuesta
    return usuario


MenuInteractivo(detector_menu, bandeja_menu, bayes_menu, compacto=True, entrada=simular(
    "8",                                                  # resumen inicial
    "1", "premios@promo-ya.com", "¡¡¡OFERTA URGENTE!!!",  # agregar un correo spam
    "Gana un premio gratis ya: http://a.co http://b.co http://c.co",
    "1", "sin-arroba",                                    # remitente inválido
    "0",                                                  # salir
)).ejecutar()

Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 8
  Total: 9 | Spam: 5 | Normales: 4 | % spam: 55.6%
Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 1
Remitente (ej. nombre@dominio.com): premios@promo-ya.com
Asunto: ¡¡¡OFERTA URGENTE!!!
Cuerpo: Gana un premio gratis ya: http://a.co http://b.co http://c.co
  Resultado: [SPAM] (21 pts) premios@promo-ya.com - ¡¡¡OFERTA URGENTE!!!
        - Palabras clave: contiene gratis, premio, urgente, oferta
        - Remitente sospechoso: remitente en lista negra (promo-ya.com)
        - Enlaces: tiene 3 enlaces
        - Mayúsculas y exclamaciones: asunto en mayúsculas, muchos signos de exclamación
        - Naive Bayes: probabilidad de spam 100%
Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige

**Sesión 2:** ver normales, entrenar Naive Bayes y **subir el umbral** (menos correos pasan a spam).


In [33]:
MenuInteractivo(detector_menu, bandeja_menu, bayes_menu, compacto=True, entrada=simular(
    "4",          # ver correos normales
    "6",          # entrenar Naive Bayes con la bandeja
    "5", "9",     # umbral = 9
    "8",          # resumen
    "0",
)).ejecutar()

Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 4
  [NORMAL] (0 pts) profe.vargas@lamolina.edu.pe - Revisión del informe
  [NORMAL] (0 pts) compa@correo.com - Reunión del grupo 5
  [NORMAL] (2 pts) secretaria@lamolina.edu.pe - URGENTE: confirma tu asistencia
        - Palabras clave: contiene urgente
  [NORMAL] (0 pts) hola@xpromo-ya.com - Boletín semanal
Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 6
  Naive Bayes aprendió de 10 correos. Entrenada: True
Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 5
Nuevo umbral (actual 5): 9
  Se analizaron 10 correos con umbral 9.
  Spam antes: 6 | Spam ahora: 3
Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 

**Sesión 3:** ver los spam que quedan con el umbral nuevo y probar entradas inválidas.

In [34]:
MenuInteractivo(detector_menu, bandeja_menu, bayes_menu, compacto=True, entrada=simular(
    "3",          # spam con umbral 9
    "5", "abc",   # umbral inválido
    "x",          # opción inexistente
    "0",
)).ejecutar()

Menú: 1 Agregar | 2 Reanalizar | 3 Ver spam | 4 Ver normales | 5 Umbral | 6 Entrenar NB | 7 Reglas | 8 Resumen | 0 Salir
Elige una opción: 3
  [SPAM] (19 pts) ventas@promo-ya.com - ¡¡¡GANASTE UN PREMIO GRATIS!!!
        - Palabras clave: contiene gratis, premio, ganaste
        - Remitente sospechoso: remitente en lista negra (promo-ya.com)
        - Enlaces: tiene 3 enlaces
        - Mayúsculas y exclamaciones: asunto en mayúsculas, muchos signos de exclamación
        - Naive Bayes: probabilidad de spam 100%
  [SPAM] (12 pts) info@ganadores.net - URGENTE: reclama tu premio
        - Palabras clave: contiene premio, urgente
        - Remitente sospechoso: remitente en lista negra (ganadores.net)
        - Naive Bayes: probabilidad de spam 100%
  [SPAM] (21 pts) premios@promo-ya.com - ¡¡¡OFERTA URGENTE!!!
        - Palabras clave: contiene gratis, premio, urgente, oferta
        - Remitente sospechoso: remitente en lista negra (promo-ya.com)
        - Enlaces: tiene 3 enlaces
        -

## 6. Informe: clases, atributos y métodos



Todos los atributos son **privados** (se escriben con doble guion bajo y Python los renombra internamente) y se leen mediante propiedades.

| Clase | Atributos privados | Métodos y propiedades públicas | Rol |
|---|---|---|---|
| `Correo` | `remitente`, `asunto`, `cuerpo`, `puntaje`, `es_spam`, `motivos` | `remitente`, `asunto`, `cuerpo`, `puntaje` (con validación), `es_spam`, `motivos`, `agregar_motivo()`, `limpiar_motivos()`, `marcar_como_spam()`, `marcar_como_normal()`, `__str__()` | Entidad que guarda los datos y el resultado del análisis |
| `Regla` (abstracta) | `nombre`, `peso` | `nombre`, `peso`, `evaluar()` *(abstracto)*, `__str__()` | Contrato común de todas las reglas |
| `ReglaPalabrasClave` | `palabras` | `evaluar()` | Palabras sospechosas en asunto y cuerpo |
| `ReglaRemitente` | `lista_negra` | `evaluar()` | Dominio del remitente en lista negra (incluye subdominios) |
| `ReglaEnlaces` | `max_enlaces` | `evaluar()` | Exceso de enlaces en el cuerpo |
| `ReglaMayusculas` | — (usa los de `Regla`) | `evaluar()` | Asunto en mayúsculas o con muchas exclamaciones |
| `ReglaNaiveBayes` | `umbral_prob`, `alfa`, `n_correos`, `total_palabras`, `frecuencias`, `vocabulario` | `entrenar()`, `entrenada`, `probabilidad_spam()`, `evaluar()` | Regla que aprende de ejemplos |
| `DetectorSpam` | `reglas`, `umbral` | `umbral` (con validación), `reglas`, `agregar_regla()`, `analizar()` | Motor que aplica las reglas y decide |
| `Bandeja` | `correos` | `correos`, `agregar()`, `listar_spam()`, `listar_normales()`, `resumen()`, `__len__()` | Almacén de correos y estadísticas |
| `MenuInteractivo` | `detector`, `bandeja`, `bayes`, `entrada`, `compacto` | `ejecutar()` | Interfaz de consola |

La celda siguiente **genera el mismo informe automáticamente** inspeccionando las clases, para comprobar que la tabla coincide con el código real:

In [35]:
import inspect


def informe_clase(cls, instancia):
    """Muestra padre, atributos de instancia y métodos/propiedades definidos en la clase."""
    padres = ", ".join(b.__name__ for b in cls.__bases__)
    atributos = [a.replace(f"_{cls.__name__}__", "").replace(f"_{Regla.__name__}__", "")
                 for a in vars(instancia)]
    metodos = []
    for nombre, miembro in vars(cls).items():
        if nombre.startswith("__") and nombre not in ("__str__", "__len__"):
            continue
        if isinstance(miembro, property):
            metodos.append(f"{nombre} (propiedad{'+setter' if miembro.fset else ''})")
        elif inspect.isfunction(miembro) and not nombre.startswith("_" + cls.__name__):
            metodos.append(f"{nombre}()")
    print(f"{cls.__name__}  (hereda de: {padres})")
    print(f"   atributos : {', '.join(atributos) or '-'}")
    print(f"   públicos  : {', '.join(metodos)}")
    if getattr(cls, "__abstractmethods__", None):
        print(f"   abstractos: {', '.join(sorted(cls.__abstractmethods__))}")
    print()


for clase, objeto in [
    (Correo, Correo("a@b.com", "x", "y")),
    (Regla, ReglaMayusculas()),            # se usa una hija solo para leer los atributos heredados
    (ReglaPalabrasClave, ReglaPalabrasClave()),
    (ReglaRemitente, ReglaRemitente()),
    (ReglaEnlaces, ReglaEnlaces()),
]:
    informe_clase(clase, objeto)

Correo  (hereda de: object)
   atributos : remitente, asunto, cuerpo, puntaje, es_spam, motivos
   públicos  : remitente (propiedad), asunto (propiedad), cuerpo (propiedad), puntaje (propiedad+setter), es_spam (propiedad), motivos (propiedad), agregar_motivo(), limpiar_motivos(), marcar_como_spam(), marcar_como_normal(), __str__()

Regla  (hereda de: ABC)
   atributos : nombre, peso
   públicos  : nombre (propiedad), peso (propiedad), evaluar(), __str__()
   abstractos: evaluar

ReglaPalabrasClave  (hereda de: Regla)
   atributos : nombre, peso, palabras
   públicos  : evaluar()

ReglaRemitente  (hereda de: Regla)
   atributos : nombre, peso, lista_negra
   públicos  : evaluar()

ReglaEnlaces  (hereda de: Regla)
   atributos : nombre, peso, max_enlaces
   públicos  : evaluar()



In [36]:
_d, _ = crear_detector()
for clase, objeto in [
    (ReglaMayusculas, ReglaMayusculas()),
    (ReglaNaiveBayes, ReglaNaiveBayes()),
    (DetectorSpam, _d),
    (Bandeja, Bandeja()),
    (MenuInteractivo, MenuInteractivo(_d, Bandeja())),
]:
    informe_clase(clase, objeto)

ReglaMayusculas  (hereda de: Regla)
   atributos : nombre, peso
   públicos  : evaluar()

ReglaNaiveBayes  (hereda de: Regla)
   atributos : nombre, peso, umbral_prob, alfa, n_correos, total_palabras, frecuencias, vocabulario
   públicos  : entrenada (propiedad), entrenar(), probabilidad_spam(), evaluar()

DetectorSpam  (hereda de: object)
   atributos : reglas, umbral
   públicos  : umbral (propiedad+setter), reglas (propiedad), agregar_regla(), analizar()

Bandeja  (hereda de: object)
   atributos : correos
   públicos  : correos (propiedad), agregar(), listar_spam(), listar_normales(), resumen(), __len__()

MenuInteractivo  (hereda de: object)
   atributos : detector, bandeja, bayes, entrada, compacto
   públicos  : ejecutar()



## 7. Conceptos de POO aplicados

| Concepto | Dónde se aplica | Evidencia en el código |
|---|---|---|
| **Clase y objeto** | `Correo`, `Bandeja`, `DetectorSpam`, cada regla | Cada correo y cada regla es un objeto con su propio estado (Pasos 1 y 2) |
| **Encapsulamiento** | Todas las clases | Atributos privados (`self.__puntaje`), lectura por `@property`, validación en los *setters* y copias de las listas internas (`motivos`, `reglas`, `correos`) |
| **Abstracción** | `Regla` | Clase abstracta (`ABC`) con `@abstractmethod evaluar()`: define **qué** debe hacer una regla sin decir **cómo**; no se puede instanciar |
| **Herencia** | `ReglaPalabrasClave`, `ReglaRemitente`, `ReglaEnlaces`, `ReglaMayusculas`, `ReglaNaiveBayes` | Heredan `nombre`, `peso` y `__str__` de `Regla` con `super().__init__(...)` |
| **Polimorfismo** | `DetectorSpam.analizar()` | Llama a `regla.evaluar(correo)` sin conocer el tipo concreto. Cinco reglas distintas responden al mismo mensaje (Paso 5) |
| **Composición** | `DetectorSpam` ◆→ `Regla`, `Bandeja` ◆→ `Correo` | Los objetos contenedores guardan listas de otros objetos |
| **Dependencia** | `DetectorSpam` ⇢ `Correo` | El detector recibe un correo, lo analiza y modifica su estado |
| **Excepciones** | Validaciones | `ValueError` (valores fuera de rango) y `TypeError` (objetos de tipo incorrecto) |
| **Métodos especiales** | `__str__`, `__len__` | Permiten `print(correo)`, `print(regla)` y `len(bandeja)` |

**Principio de diseño (abierto/cerrado):** para agregar una regla nueva basta con crear una subclase de `Regla`, implementar `evaluar()` y registrarla con `agregar_regla()`. **`DetectorSpam` no se modifica.** `ReglaNaiveBayes` es la prueba: se integró al sistema sin tocar el detector.

**Separación de responsabilidades:** `Correo` guarda datos, las reglas evalúan, `DetectorSpam` decide, `Bandeja` almacena y `MenuInteractivo` es la única clase que interactúa con el usuario.


## 8. Conclusiones